In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle

from tqdm import tqdm

In [2]:
# Total number of data points: 18.752.333
# Number of slices: 143.680
# ~ 130 data points per slice [6, 499]

#       Date	                    Texp	        size
# count	143680	                    143680.000000	143680.000000
# mean	2016-10-28  16:46:33.407572	0.540287	    130.514567
# min	2000-01-03  00:00:00	    0.002738	    6.000000
# 25%	2012-12-19  00:00:00	    0.071184	    62.000000
# 50%	2018-03-27  00:00:00	    0.238193	    115.000000
# 75%	2022-02-28  00:00:00	    0.750171	    170.000000
# max	2024-12-31  00:00:00	    6.048000	    499.000000
# std	NaN	                        0.755189	    89.166256

df = pd.read_parquet("data/ivdata.parquet")
df.head()

,Date,Expiry,Texp,z,w
219043,2000-01-03,2000-01-22,0.052019,-0.400915,NaN
219044,2000-01-03,2000-01-22,0.052019,-0.350905,0.000097
219045,2000-01-03,2000-01-22,0.052019,-0.326807,0.000086
219046,2000-01-03,2000-01-22,0.052019,-0.303277,0.000076
219047,2000-01-03,2000-01-22,0.052019,-0.280287,0.000066


In [3]:
# Group data into maturity bins.
bins = np.array([
    -np.inf, 
    7,
    30, 
    90,
    180, 
    365, 
    # np.inf
    ])

labels= [
    "<=1wk",    # 0-7 days
    "1m",       # 7-30 days
    "3m",       # 30-90 days
    "6m",       # 90 - 180 days
    "1y",       # 180 - 365 days
    # ">1y"       # 365+ days
]

# Calculate days to expiry
df["DTE"] =  (df["Expiry"] - df["Date"]).dt.days

df["Maturity-Group"] = pd.cut(
    df["DTE"],
    bins=list(bins),
    labels=labels,
    include_lowest=True
)
df.head()

,Date,Expiry,Texp,z,w,DTE,Maturity-Group
219043,2000-01-03,2000-01-22,0.052019,-0.400915,NaN,19,1m
219044,2000-01-03,2000-01-22,0.052019,-0.350905,0.000097,19,1m
219045,2000-01-03,2000-01-22,0.052019,-0.326807,0.000086,19,1m
219046,2000-01-03,2000-01-22,0.052019,-0.303277,0.000076,19,1m
219047,2000-01-03,2000-01-22,0.052019,-0.280287,0.000066,19,1m


In [4]:
for mi, (maturityLabel, maturityGroup) in enumerate(df.groupby("Maturity-Group")):
    maturityGroup.to_parquet(f"group_{mi}.parquet")

In [ ]:
def format_bytes(n_bytes):
    units = ["B", "KB", "MB", "GB", "TB", "PB"]

    size = float(n_bytes)

    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024
        
def durrleman(x, y):
    dy = np.gradient(y, x)
    d2y = np.gradient(dy, x)
    
    return ((1 - x * dy / (2 * y)) ** 2 - dy ** 2 / 4 * (1 / y + 1 / 4) + d2y / 2)[2:-2]

def durr_neg_area(y):
    mask = y < 0
    A = np.trapezoid(np.abs(y))
    
    if A == 0:
        return 0    
    
    return np.trapezoid(-y[mask]) / A

In [ ]:
import joblib
from itertools import combinations

from scipy.interpolate import interp1d
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

min_points_per_slice = 20
num_grid_points = 100
num_clusters_per_group = 4

zgrid = np.linspace(-1, 1, num_grid_points)


def tril_index(i: int, j: int, n: int) -> int:
    """Map (i, j) with j > i to a flat lower-triangular index."""
    return n * i - i * (i + 1) // 2 + (j - i - 1)


def preprocess(z: np.ndarray, w: np.ndarray):
    """Preprocess z and w for a single slice. Replace with your own logic."""
    # z_norm = z / np.abs(z).max()
    # w_norm = w

    return z, w


def distance(z_i, w_i, z_j, w_j) -> float:
    """Compute distance between two preprocessed (z, w) slices."""
    return float(np.linalg.norm(z_i - z_j))

def ext_dist(z_i, w_i, z_j, w_j) -> float:
    z_min = max(z_i.min(), z_j.min())
    z_max = min(z_i.max(), z_j.max())
    
    if z_max <= z_min:
        return 0
    
    z_grid = np.linspace(z_min, z_max, num_grid_points)
    w_i_interp = np.interp(z_grid, z_i, w_i)
    w_j_interp = np.interp(z_grid, z_j, w_j)
    
    return float(np.linalg.norm(w_i_interp - w_j_interp))

def _compute_pair(i: int, j: int, slices: list) -> tuple:
    """Worker: preprocess slice i and j, return (i, j, dist)."""
    z_i, w_i = preprocess(*slices[i])
    z_j, w_j = preprocess(*slices[j])
    dist = ext_dist(z_i, w_i, z_j, w_j)
    return i, j, dist

for mi, (maturityLabel, maturityGroup) in enumerate(df.groupby("Maturity-Group")):

    # Collect all (date, texp) slices upfront — these are the matrix rows/cols
    slices = []       # list of (z, w) arrays, one per (date, texp)
    slice_keys = []   # matching (date, texp) labels for bookkeeping

    for (date, texp), group in maturityGroup.groupby(["Date", "Texp"]):
        group = group.dropna(subset=["z", "w"])
        if group.empty or len(group) < min_points_per_slice:
            continue
        
        z = group["z"].to_numpy(dtype=np.float32)
        w = group["w"].to_numpy(dtype=np.float32)
        
        if np.abs(z).max() == 0:
            continue
        
        slices.append((z, w))
        slice_keys.append((date, texp))

    n = len(slices)
    if n < 2:
        continue

    flat_size = n * (n - 1) // 2
    matrix_path = f"data/distance/matrix_group_{mi}.dat"
    matrix = np.memmap(matrix_path, dtype=np.float16, mode="w+", shape=(flat_size,))

    pairs = list(combinations(range(n), 2))

    results = joblib.Parallel(n_jobs=-1, batch_size="auto")(
        joblib.delayed(_compute_pair)(i, j, slices)
        for i, j in tqdm(pairs, desc=f"Group {mi} ({n} slices, {len(pairs)} pairs)")
    )

    for i, j, dist in results:
        matrix[tril_index(i, j, n)] = dist

    matrix.flush()

    np.save(f"matrix_group_{mi}_keys.npy", np.array(slice_keys, dtype=object))

In [ ]:

from scipy.interpolate import interp1d
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

min_points_per_slice = 20
num_grid_points = 100
num_clusters_per_group = 4

zgrid = np.linspace(-1, 1, num_grid_points)
results_list = []

data_matrix = None
cluster_array = None

R = {}

for mLabel, maturityDf in  df.groupby("Maturity-Group"):
    feature_vectors = []
    group_keys = []

    arb_list = []

    skipped_slices = 0
    arb = 0
    arbval = 0
    
    for (date, texp), group in tqdm(maturityDf.groupby(["Date","Texp"]), desc=f"Maturity {mLabel}:"):
        # Remove NaN values
        group = group.dropna(subset=["z", "w"])
        mask = group["z"] < 0
        
        # Ignore slice if it has to few data points
        if len(group) < min_points_per_slice:
            continue
        
        # Normalise values into [-1, 1]
        z_min = abs(group.loc[mask, "z"].min())
        z_max = group.loc[~mask, "z"].max()
        
        # Skip slices that cannot be normalized
        if not (z_min > 0 and z_max > 0):
            # print("Skipping", date, texp)
            skipped_slices += 1
            continue
        
        z_norm = group["z"] / z_min
        z_norm[~mask] = group.loc[~mask, "z"] / z_max
        
        # Interpolate to common grid
        w_fixed = np.interp(zgrid, z_norm, group["w"])
        
        durr = durrleman(zgrid, w_fixed)
        alph = durr_neg_area(durr)
        
        if (durr < 0).any():
            arbval += alph
            arb += 1
            arb_list.append(1)
        else:
            arb_list.append(0)
        
        feature_vectors.append(w_fixed)
        group_keys.append((date, texp))
    
    print("ARB", arb, arbval / arb)
    
    X = np.array(feature_vectors)

    print("Skipped Slices", skipped_slices)
    print(f"Feature Matrix size: {X.shape}, {format_bytes(X.nbytes)} bytes")
    print(f"Distance Matrix size: ({X.shape[0]}, {X.shape[0]}), {format_bytes(X.shape[0] ** 2 * 8)} bytes")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
     
    mbk = MiniBatchKMeans(
        n_clusters=num_clusters_per_group, 
        batch_size=10_024, 
        random_state=42, 
        n_init="auto"
    )
    cluster_labels = mbk.fit_predict(X_scaled)
    results = pd.DataFrame(group_keys, columns=['Date', 'Texp'])
    results['Cluster'] = cluster_labels
    
    results_list.append(results)
    
    R[mLabel] = {
        "labels": cluster_labels,
        "data": X,
        "keys": group_keys,
        "arb": arb_list
    }

with open("data/cluster/res.pkl", "wb") as f:
    pickle.dump(R, f)

In [ ]:
with open("data/cluster/res.pkl", "rb") as file:
    R = pickle.load(file)

In [ ]:
mat_labels= [
    "<=1wk",    # 0-7 days
    "1m",       # 7-30 days
    "3m",       # 30-90 days
    "6m",       # 90 - 180 days
    "1y",       # 180 - 365 days
    ">1y"       # 365+ days
]



for i, ml in enumerate(mat_labels):
    data = R[ml]["data"]
    labels = R[ml]["labels"]
    keys = R[ml]["keys"]

    L = np.unique(labels)
    for cluster in tqdm(L, total=len(L)):
        T = np.array(keys)[labels==cluster][:, 1]

        plt.figure()

        for _, g in df[df["Texp"].isin(T)].groupby(["Date", "Texp"]):
            z = g["z"]
            w = g["w"]
            plt.plot(z, w, label=f"Cluster {cluster}")
            
        plt.title(f"Group: {ml}, Cluster {cluster}")
        
        plt.savefig(f"images/ca/{i}_{cluster}.png")
        plt.close()

In [ ]:
min_points_per_slice = 20
num_grid_points = 100
num_clusters_per_group = 4

zgrid = np.linspace(-1, 1, num_grid_points)
#%matplotlib qt
data = R[">1y"]["data"]
labels = R[">1y"]["labels"]
# arb_slices = data[np.array(arb_labels) == 1, :]
for l in range(277):
    y = data[labels==3][l, :]
    plt.plot(zgrid,y)
    # plt.plot(zgrid, data.mean(axis=0))
# plt.axhline(0, -1, 1)
plt.show()

In [ ]:
zgrid

In [ ]:
# data_matrix = np.load("data/cluster/data.npy")
# cluster_array = np.load("data/cluster/labels.npy", allow_pickle=True)
# labels = cluster_array[:, 2]

In [ ]:
def plot_cluster_profiles(X, labels, title, filename, x=None, show_individual=False):
    """
    Plot cluster means and variation bands.

    Parameters
    ----------
    X : ndarray of shape (M, N)
        Matrix of M profiles with N points each.

    labels : ndarray of shape (M,)
        Cluster labels for each profile.

    x : ndarray of shape (N,), optional
        X-axis values. If None, uses 0, ..., N-1.

    show_individual : bool, default=False
        Whether to plot all individual profiles in the background.
    """

    if x is None:
        x = np.arange(X.shape[1])

    clusters = np.sort(np.unique(labels))
    n_clusters = len(clusters)

    n_cols = min(3, n_clusters)
    n_rows = int(np.ceil(n_clusters / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        sharex=True,
        # sharey=True
    )

    axes = np.atleast_1d(axes).flatten()

    for ax, cluster in zip(axes, clusters):

        cluster_data = X[labels == cluster]

        if len(cluster_data) == 0:
            ax.set_title(f"Cluster {cluster} (empty)")
            continue

        mean = cluster_data.mean(axis=0)
        std = cluster_data.std(axis=0)

        if show_individual:
            for curve in cluster_data:
                ax.plot(x, curve, alpha=0.1)

        ax.fill_between(
            x,
            mean - std,
            mean + std,
            alpha=0.3,
            label="±1 std"
        )

        ax.plot(
            x,
            mean,
            linewidth=2,
            label="Mean"
        )

        ax.set_title(
            f"Cluster {cluster}\nN={len(cluster_data)}"
        )

        ax.grid(alpha=0.3)

        if cluster == clusters[0]:
            ax.legend()

    # Remove unused axes
    for ax in axes[len(clusters):]:
        fig.delaxes(ax)

    fig.supxlabel("Profile coordinate")
    fig.supylabel("Value")

    fig.suptitle(f'Cluster Group: {title}')

    plt.tight_layout()
    plt.savefig(f"images/clusters/{filename}")
    plt.show()

In [ ]:
for i, (mLabel, result) in enumerate(R.items()):
    labels = result["labels"]
    data = result["data"]
    plot_cluster_profiles(data, labels, mLabel, f"Cluster_group_{i}.png")
    